In [1]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Role,
    WorkflowBuilder,
    WorkflowContext,
    ai_function,
    executor,
)

# 🤖 GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

✅ All imports successful!


## ជំហានទី 1: កំណត់ម៉ូឌែល Pydantic សម្រាប់លទ្ធផលមានរចនាសម្ព័ន្ធ

ម៉ូឌែលទាំងនេះកំណត់ **រចនាសម្ព័ន្ធ** ដែលភ្នាក់ងារនឹងត្រឡប់មក។ ការប្រើ `response_format` ជាមួយ Pydantic ធានាថា:
- ✅ ការទាញយកទិន្នន័យដែលមានប្រភេទសុវត្ថិភាព
- ✅ ការផ្ទៀងផ្ទាត់ដោយស្វ័យប្រវត្តិ
- ✅ គ្មានកំហុសក្នុងការវិភាគពីចម្លើយអត្ថបទសេរី
- ✅ ការបញ្ជាទិសដោយលក្ខខណ្ឌយ៉ាងងាយស្រួលដោយផ្អែកលើវាល


In [2]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)


## ជំហាន 2: បង្កើតឧបករណ៍កក់សណ្ឋាគារ

ឧបករណ៍នេះគឺជាអ្វីដែល **availability_agent** នឹងហៅដើម្បីពិនិត្យថាតើបន្ទប់អាចប្រើបានឬទេ។ 

យើងប្រើ `@ai_function` decorator ដើម្បី:
- បម្លែងមុខងារ Python មួយទៅជា ឧបករណ៍ដែលអាចហៅដោយ AI
- បង្កើត JSON schema ស្វ័យប្រវត្តិសម្រាប់ LLM
- ដោះស្រាយការផ្ទៀងផ្ទាត់ប៉ារ៉ាម៉ែត្រ
- អនុញ្ញាតឲ្យភ្នាក់ងារហៅស្វ័យប្រវត្តិ

សម្រាប់ការបង្ហាញនេះ:
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → មានបន្ទប់ ✅
- **ទីក្រុងផ្សេងទៀតទាំងអស់** → មិនមានបន្ទប់ ❌


In [3]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## ជំហាន 3: កំណត់មុខងារលក្ខខណ្ឌសម្រាប់ការតម្រង

មុខងារទាំងនេះពិនិត្យមើលការឆ្លើយតបរបស់ភ្នាក់ងារ ហើយកំណត់ថាត្រូវយកផ្លូវណាមួយនៅក្នុងលំហូរការងារ។

**លំនាំសំខាន់:**
1. ពិនិត្យថាតើសារ គឺជា `AgentExecutorResponse` ឬទេ
2. វិភាគលទ្ធផលដែលមានរចនាសម្ព័ន្ធ (ម៉ូឌែល Pydantic)
3. ផ្ដល់តម្លៃ `True` ឬ `False` ដើម្បីគ្រប់គ្រងការតម្រង

លំហូរការងារនេះនឹងវាយតម្លៃលក្ខខណ្ឌទាំងនេះនៅលើ **edges** ដើម្បីសម្រេចថាតើត្រូវហៅ executor ណាដើម្បីអនុវត្តបន្ទាប់។


In [4]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)


## ជំហានទី 4: បង្កើត Executor ផ្ទាល់ខ្លួនសម្រាប់បង្ហាញ

Executors គឺជាសមាសធាតុក្នុងដំណើរការការងារ ដែលអនុវត្តការបម្លែង ឬបង្កើតផលប៉ះពាល់ខាងក្រៅ។ យើងប្រើ `@executor` decorator ដើម្បីបង្កើត executor ផ្ទាល់ខ្លួន ដែលបង្ហាញលទ្ធផលចុងក្រោយ។

**គំនិតសំខាន់ៗ:** 
- `@executor(id="...")` - ចុះបញ្ជីមុខងារជា executor សម្រាប់ដំណើរការការងារ
- `WorkflowContext[Never, str]` - សញ្ញាប្រាប់ប្រភេទសម្រាប់ការបញ្ចូល/ការបញ្ចេញ
- `ctx.yield_output(...)` - ផ្តល់លទ្ធផលចុងក្រោយនៃដំណើរការការងារ


In [5]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

✅ display_result executor created with @executor decorator


## ជំហានទី 5: ផ្ទុកអថេរបរិស្ថាន

កំណត់រចនាសម្ព័ន្ធកម្មវិធីអតិថិជន LLM។ ឧទាហរណ៍នេះអាចប្រើបានជាមួយ:
- **GitHub Models** (កម្រិតឥតគិតថ្លៃ ជាមួយ GitHub token)
- **Azure OpenAI**
- **OpenAI**


In [6]:
# Load environment variables
load_dotenv()

# Check for GitHub Models or OpenAI
chat_client = OpenAIChatClient(base_url=os.environ.get(
    "GITHUB_ENDPOINT"), api_key=os.environ.get("GITHUB_TOKEN"), model_id="gpt-4o")

## ជំហាន 6: បង្កើតភ្នាក់ងារ AI ជាមួយលទ្ធផលដែលមានរចនាសម្ព័ន្ធ

យើងបង្កើត **ភ្នាក់ងារពិសេសចំនួនបី** ដែលនីមួយៗត្រូវបានដាក់នៅក្នុង `AgentExecutor`:

1. **availability_agent** - ពិនិត្យភាពមានបន្ទប់សណ្ឋាគារដោយប្រើឧបករណ៍
2. **alternative_agent** - ផ្តល់យោបល់អំពីទីក្រុងជំនួស (ពេលគ្មានបន្ទប់)
3. **booking_agent** - លើកទឹកចិត្តឱ្យកក់ (ពេលមានបន្ទប់)

**លក្ខណៈសំខាន់៖**
- `tools=[hotel_booking]` - ផ្ដល់ឧបករណ៍ទៅភ្នាក់ងារ
- `response_format=PydanticModel` - បង្ខាំងឱ្យចេញលទ្ធផល JSON ដែលមានរចនាសម្ព័ន្ធ
- `AgentExecutor(..., id="...")` - បិទជុំភ្នាក់ងារ​សម្រាប់ការប្រើប្រាស់ក្នុងលំហូរការងារ


In [7]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## ជំហានទី 7: បង្កើត Workflow ជាមួយចំណុចភ្ជាប់ដែលមានលក្ខខណ្ឌ

ឥឡូវនេះយើងប្រើ `WorkflowBuilder` ដើម្បីកសាងក្រាហ្វជាមួយការបញ្ជូនដែលមានលក្ខខណ្ឌ:

**រចនាសម្ព័ន្ធ Workflow:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**មុខងារសំខាន់ៗ:**
- `.set_start_executor(...)` - កំណត់ចំណុចចូល
- `.add_edge(from, to, condition=...)` - បន្ថែមចំណុចភ្ជាប់ដែលមានលក្ខខណ្ឌ
- `.build()` - បញ្ចប់ការកសាង Workflow


In [8]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ជំហាន 8: បើកដំណើរការករណីតេស្ត 1 - ទីក្រុងដែលគ្មានបន្ទប់ទំនេរ (Paris)

យើងនឹងសាកល្បងផ្លូវ **គ្មានបន្ទប់ទំនេរ** ដោយស្នើសុំសណ្ឋាគារនៅទីក្រុង Paris (ដែលក្នុងការត្រួតពិនិត្យរបស់យើងមិនមានបន្ទប់ទេ).


In [9]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ជំហានទី 9: រត់ករណីតេស្ត 2 - ទីក្រុង ដែលមានភាពអាចរកបាន (Stockholm)

ឥឡូវនេះ យើងនឹងសាកល្បងផ្លូវ **ភាពអាចរកបាន** ដោយស្នើសុំព័ត៌មានសណ្ឋាគារនៅ Stockholm (ដែលមានបន្ទប់នៅក្នុងសិមូឡេស្យុងរបស់យើង).


In [10]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ចំនុចសំខាន់ និងជំហានបន្ទាប់

### ✅ អ្វីដែលអ្នកបានរៀន:

1. **WorkflowBuilder លំនាំ**
   - ប្រើ `.set_start_executor()` ដើម្បីកំណត់ចំណុចចូល
   - ប្រើ `.add_edge(from, to, condition=...)` សម្រាប់ការតម្រួតផ្លូវដោយលក្ខខណ្ឌ
   - ហៅ `.build()` ដើម្បីបញ្ចប់ workflow

2. **ការបញ្ជូនដោយលក្ខខណ្ឌ**
   - មុខងារលក្ខខណ្ឌពិនិត្យ `AgentExecutorResponse`
   - វិភាគលទ្ធផលដែលមានរចនាសម្ព័ន្ធ ដើម្បីសម្រេចការបញ្ជូន
   - ត្រឡប់ `True` ដើម្បីសកម្ម edge, `False` ដើម្បីរំលងវា

3. **ការរួមបញ្ចូលឧបករណ៍**
   - ប្រើ `@ai_function` ដើម្បីបម្លែងអនុគមន៍ Python ទៅជាឧបករណ៍ AI
   - Agents ហៅឧបករណ៍ដោយស្វ័យប្រវត្តិពេលត្រូវការ
   - ឧបករណ៍ត្រឡប់ JSON ដែល agents អាចវិភាគបាន

4. **លទ្ធផលដែលមានរចនាសម្ព័ន្ធ**
   - ប្រើគំរូ Pydantic សម្រាប់ការដកទិន្នន័យប្រកបដោយប្រភេទ
   - កំណត់ `response_format=MyModel` ពេលបង្កើត agents
   - វិភាគចម្លើយដោយប្រើ `Model.model_validate_json()`

5. **Executor ផ្ទាល់ខ្លួន**
   - ប្រើ `@executor(id="...")` ដើម្បីបង្កើតធាតុក្នុង workflow
   - Executors អាចផ្លាស់ប្តូរទិន្នន័យ ឬអនុវត្តផលប៉ះពាល់ក្រោយ
   - ប្រើ `ctx.yield_output()` ដើម្បីផលិតលទ្ធផលនៃ workflow

### 🚀 ករណីប្រើប្រាស់ក្នុងពិភពពិត:

- **Travel Booking**: ពិនិត្យភាពអាចមាន, ស្នើជម្រើសជំនួស, ប្រៀបធៀបជម្រើស
- **Customer Service**: បញ្ជូនអាស្រ័យលើប្រភេទបញ្ហា, អារម្មណ៍, អាទិភាព
- **E-commerce**: ពិនិត្យស្តុក, ស្នើជម្រើសជំនួស, ដំណើរការបញ្ជាទិញ
- **Content Moderation**: បញ្ជូនដោយផ្អែកលើពិន្ទុពុល និងស្លាករបស់អ្នកប្រើ
- **Approval Workflows**: បញ្ជូនដោយផ្អែកលើចំនួន, តួនាទីអ្នកប្រើ, កម្រិតហានិភ័យ
- **Multi-stage Processing**: បញ្ជូនដោយផ្អែកលើគុណភាពទិន្នន័យ និងភាពពេញលេញ

### 📚 ជំហានបន្ទាប់:

- បន្ថែមលក្ខខណ្ឌស្មុគស្មាញ (លក្ខខណ្ឌច្រើន)
- អនុវត្តលាប់ជាមួយការគ្រប់គ្រងស្ថានភាព workflow
- បន្ថែម sub-workflows សម្រាប់ធាតុដែលអាចប្រើឡើងវិញ
- រួមបញ្ចូលជាមួយ real APIs (hotel booking, inventory systems)
- បន្ថែមការដោះស្រាយកំហុស និងផ្លូវជំនួស
- បង្ហាញទម្រង់ workflow ដោយប្រើឧបករណ៍បង្ហាញដែលមានរួច


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការសេចក្តីប្រកាសអត់ទទួលខុសត្រូវ**:
ឯកសារ​នេះ​ត្រូវ​បាន​បកប្រែ​ដោយ​ប្រើ​សេវាកម្ម​ប្រែ​ភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator). ខណៈពេល​យើង​ខិតខំ​ប្រឹងប្រែង​ដើម្បី​ភាព​ត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថា ការប្រែដោយប្រព័ន្ធស្វ័យប្រវត្តិ​អាចមានកំហុស ឬភាពមិន​ត្រឹមត្រូវ។ សូមយោងទៅឯកសារ​ដើម​ក្នុងភាសាដើមជា​ប្រភពព័ត៌មាន​ដែលត្រូវបានទុកចិត្ត។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមផ្តល់អនុសាសន៍ឲ្យមានការបកប្រែ​ដោយអ្នកបកប្រែវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការពន្យល់ខុសណាមួយដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
